# 22 - llama.cpp baseline (CUDA build for sm_75, f16 GGUF)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release
!cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server
!hf download bartowski/Qwen2.5-1.5B-Instruct-GGUF --include "*f16.gguf" --local-dir /tmp/gguf
!ls -la /tmp/gguf

If cmake cannot find CUDA add `-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc`. If no f16 file was downloaded: `pip install gguf sentencepiece`, then `python /tmp/llama.cpp/convert_hf_to_gguf.py <HF dir> --outtype f16 --outfile /tmp/gguf/qwen2.5-1.5b-f16.gguf`.

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/llama.log", "w")
p = subprocess.Popen(["/tmp/llama.cpp/build/bin/llama-server", "-m", "/tmp/gguf/<file>.gguf", "-ngl", "99", "-c", "65536", "-np", "16", "--port", "8002", "-fa", "on"], stdout=log, stderr=subprocess.STDOUT)
# if '-fa on' is rejected, use '-fa'